训练过程页按启用章节连续编号；可在 `to_excel(training_sheet_index=...)` 中指定。重要性使用验证集 SHAP；树模型同时保留原生重要性，两者不混合尺度。SHAP 的 permutation 指排列式 Shapley 近似，不是打乱特征后比较 AUC 的性能置换重要性。

# 全模型建模链路与报告验收

使用 970 行真实放款数据验证：数据划分、特征筛选、基线对比、两点网格搜索、最终模型、完整 Excel 报告及保存恢复。
分类目标为 FPD，保持 0/1 二分类；五种 net 另验证放款金额回归。报告生成开启图表、金额、日期、商品类别与样例记录，并采用 strict 模式。

这里的小规模搜索用于接口和数值验收，不能据此认定模型已经达到业务上线标准。模型效果直接与均值/常量基准和原始行指标对照，不要求每种模型都优于基准。
需要当前源码的 `[net,boost,tune]` 依赖。TabPFN 的真实验收使用 9.2.0 与本地官方 v2 权重，路径可由环境变量指定。

## 1. 环境与输入

默认从 examples/hscredit_yyp.xlsx 读取，输出到 artifacts/all_model_workflow。
支持 HSCREDIT_EXAMPLE_INPUT、HSCREDIT_EXAMPLE_OUTPUT，以及 HSCREDIT_TABPFN_CLASSIFIER_PATH / HSCREDIT_TABPFN_REGRESSOR_PATH。
HSCREDIT_MODEL_CASES 可用于修复后的指定案例复核，留空执行全部案例；每次都保存实际运行清单。

In [ ]:
from pathlib import Path
from importlib.metadata import version
from datetime import date
import os, sys, json, time, traceback, gc
import numpy as np
import pandas as pd
import torch
import optuna
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from openpyxl import load_workbook
import hscredit as hc
from hscredit.report import ModelReport, auto_model_report, compare_models
from hscredit.excel import dataframe2excel
from hscredit.core.rules import Rule
from hscredit.core.models._lifecycle import atomic_save_pickle
from hscredit.utils.serialization import load_verified_pickle

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'hscredit').is_dir())
input_path = Path(os.environ.get('HSCREDIT_EXAMPLE_INPUT', root / 'examples/hscredit_yyp.xlsx'))
output = Path(os.environ.get('HSCREDIT_EXAMPLE_OUTPUT', root / 'artifacts/all_model_workflow'))
output.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
optuna.logging.set_verbosity(optuna.logging.WARNING)
environment = {'日期': str(date.today()), '解释器': sys.executable, 'Python': sys.version, '依赖': {k: version(k) for k in ['numpy','pandas','scikit-learn','torch','tabpfn','tabm','xgboost','lightgbm','catboost','ngboost','optuna']}}
(output / '运行环境.json').write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(environment['依赖'].items(), columns=['依赖','版本']))

## 2. 按完整日期划分训练、选参验证和独立测试

切分点取唯一放款日期的 60% 和 80% 位置，同一天不会分散到两个集合。只使用约定的三个申请时特征，排除客户号、FPD、逾期字段和放款金额。
所有填补、缩放、筛选均由训练数据拟合；独立测试不参与模型比较或选参。

In [ ]:
data = pd.read_excel(input_path).sort_values('放款时间', kind='stable')
features = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '青云24']
days = data['放款时间'].drop_duplicates().tolist()
validation_start, test_start = days[int(len(days)*.6)], days[int(len(days)*.8)]
train = data.loc[data['放款时间'] < validation_start].copy()
valid = data.loc[(data['放款时间'] >= validation_start) & (data['放款时间'] < test_start)].copy()
test = data.loc[data['放款时间'] >= test_start].copy()
datasets = {'训练集': train, '选参验证集': valid, '独立测试集': test}
report_source = f'{input_path.name}，{len(data)}行'
def report_description(task):
    return ('FPD预测' if task=='binary' else '金额回归')+f'；验证{validation_start:%Y-%m-%d}起，独立测试{test_start:%Y-%m-%d}起。'
assert sum(map(len, datasets.values())) == len(data)
assert train['放款时间'].max() < valid['放款时间'].min() < test['放款时间'].min()
assert valid['放款时间'].max() < test['放款时间'].min()
assert not set(train.index).intersection(valid.index) and not set(train.index).intersection(test.index)
assert all(set(frame.FPD) == {0, 1} for frame in datasets.values())
split_table = pd.DataFrame([{'数据集': name, '样本数': len(frame), '坏样本数': int(frame.FPD.sum()), '坏样本率': frame.FPD.mean(), '开始时间': frame['放款时间'].min(), '结束时间': frame['放款时间'].max()} for name,frame in datasets.items()])
display(split_table)
split_table.to_csv(output / '数据划分.csv', index=False, encoding='utf-8-sig')

## 3. 模型清单、筛选器和真实搜索空间

普通分类模型使用评分卡粗筛；回归使用方差筛选。填补、缩放与筛选放入 Pipeline，各试验独立拟合。
ScoreCard/RoundScoreCard 使用未拟合分箱器和 raw 输入，每次克隆后仅在本次训练划分学习分箱。
校准四种方法分别运行。CalibratedModel、评分转换器及漂移校准等组合组件在后续附加分支核对。

In [ ]:
BASE_NAMES = ['LogisticRegression','RandomForest','ExtraTrees','GradientBoosting','SVM','DecisionTreeClassifier','XGBoost','LightGBM','CatBoost','NGBoost','ResNet','FTTransformer','TabM','TabPFN','TabNet','ScoreCard','RoundScoreCard','ProbabilityScoreCard','RulesClassifier']
CASES = [(name,'binary') for name in BASE_NAMES] + [(f'ProbabilityCalibrator-{method}','binary') for method in ['platt','isotonic','beta','histogram']]
CASES += [(name,'regression') for name in ['ResNet','FTTransformer','TabM','TabPFN','TabNet']]
CASES += [('TabPFN-finetune','binary'),('TabPFN-finetune','regression')]

def make_case(name, task):
    target = 'FPD' if task == 'binary' else '放款金额'
    selector = (hc.ScorecardFeatureSelection(target=target, target_rm=True, n_jobs=1, iv_threshold=.02, corr_threshold=.85, bin_params={'method':'quantile','max_n_bins':4,'monotonic':False}) if task == 'binary' else hc.VarianceSelector(target=target, target_rm=True, threshold=0., n_jobs=1))
    common = {'target':target, 'random_state':42}
    fit_params = {}
    if name == 'LogisticRegression':
        estimator = hc.LogisticRegression(max_iter=2000, calculate_stats=True, **common)
        space = {'C':[.2,2.]}
    elif name in ['RandomForest','ExtraTrees','GradientBoosting']:
        estimator = getattr(hc,name)(n_estimators=100, max_depth=3, n_jobs=1, **common)
        space = {'max_depth':[2,4]}
    elif name == 'SVM':
        estimator = hc.SVM(n_jobs=1, **common)
        space = {'C':[.2,2.]}
    elif name == 'DecisionTreeClassifier':
        estimator = hc.DecisionTreeClassifier(n_jobs=1, max_depth=3, **common)
        space = {'max_depth':[2,4]}
    elif name in ['XGBoost','LightGBM','NGBoost']:
        options = {'max_depth':3} if name == 'XGBoost' else {'verbosity':-1,'min_child_samples':8} if name == 'LightGBM' else {'natural_gradient':False, 'base_max_depth':2}
        estimator = getattr(hc,name)(n_estimators=100, n_jobs=1, early_stopping_rounds=None, **options, **common)
        space = {'learning_rate':[.025,.1]}
    elif name == 'CatBoost':
        estimator = hc.CatBoost(iterations=100, depth=3, n_jobs=1, early_stopping_rounds=None, allow_writing_files=False, **common)
        space = {'depth':[2,4]}
    elif name in ['ResNet','FTTransformer','TabM']:
        options = {'k':4} if name == 'TabM' else {}
        estimator = getattr(hc,name)(objective=task, d_block=32, n_blocks=2, max_epochs=30, batch_size=128, early_stopping_rounds=5, device='cpu', **options, **common)
        space = {'learning_rate':[.0003,.003]}
    elif name == 'TabNet':
        estimator = hc.TabNet(objective=task, n_d=16, n_a=16, n_steps=3, max_epochs=30, batch_size=128, early_stopping_rounds=5, device='cpu', **common)
        space = {'learning_rate':[.005,.02]}
    elif name.startswith('TabPFN'):
        key = 'HSCREDIT_TABPFN_CLASSIFIER_PATH' if task == 'binary' else 'HSCREDIT_TABPFN_REGRESSOR_PATH'
        local = root / '.audit_tmp/net-models/weights' / ('tabpfn-v2-classifier.ckpt' if task == 'binary' else 'tabpfn-v2-regressor.ckpt')
        path = os.environ.get(key, str(local) if local.exists() else 'auto')
        finetune = name.endswith('finetune')
        estimator = hc.TabPFN(objective=task, n_estimators=1, model_path=path, device='cuda' if torch.cuda.is_available() else 'cpu', n_jobs=1, inference_precision=torch.float32, finetune=finetune, finetune_params={'model_version':'v2','max_epochs':1,'batch_size':128,'validation_fraction':0,'early_stopping_rounds':None} if finetune else None, **common)
        space = {'finetune_params__learning_rate':[.000003,.00001]} if finetune else {'softmax_temperature':[.8,1.]}
    elif name in ['ScoreCard','RoundScoreCard']:
        binner = hc.OptimalBinning(method='quantile', max_n_bins=4, monotonic=False, n_jobs=1)
        estimator = getattr(hc,name)(target=target, binner=binner, lr_model=hc.LogisticRegression(max_iter=2000, calculate_stats=True, random_state=42))
        space = {'lr_model__C':[.2,2.]}
        fit_params = {'模型__input_type':'raw'}
    elif name == 'ProbabilityScoreCard':
        estimator = hc.ProbabilityScoreCard(model=hc.LogisticRegression(max_iter=2000, calculate_stats=True, random_state=42), prefit=False, target=target)
        space = {'model__C':[.2,2.]}
    elif name == 'RulesClassifier':
        rules = [Rule(f'`{features[0]}` < {train[features[0]].median()}'), Rule(f'`{features[1]}` > {train[features[1]].median()}')]
        estimator = hc.RulesClassifier(rules=rules, target=target, logic='or', n_jobs=1)
        space = {'logic':['or','and']}
    else:
        method = name.split('-')[1]
        estimator = hc.ProbabilityCalibrator(model=hc.LogisticRegression(max_iter=2000, calculate_stats=False, random_state=42), target=target, method=method, calib_ratio=.25, random_state=42)
        space = {'model__C':[.2,2.]}
    steps = [('特征筛选',selector)]
    if name not in ['ScoreCard','RoundScoreCard','RulesClassifier']:
        steps.append(('缺失填补',SimpleImputer(strategy='median').set_output(transform='pandas')))
    if name in ['LogisticRegression','SVM','ProbabilityScoreCard'] or name.startswith('ProbabilityCalibrator'):
        steps.append(('标准化',StandardScaler().set_output(transform='pandas')))
    steps.append(('模型',estimator))
    return Pipeline(steps), {'模型__'+key:value for key,value in space.items()}, fit_params

chosen = os.environ.get('HSCREDIT_MODEL_CASES','')
cases = [case for case in CASES if not chosen or f'{case[0]}-{case[1]}' in chosen.split(',')]
assert cases, '未选择有效模型案例'
display(pd.DataFrame(cases, columns=['模型','任务']).replace({'任务':{'binary':'二分类','regression':'回归'}}))

## 4. 基线训练与模型对比

模型对比只使用训练集和选参验证集。分类、回归分别对比，避免混合 AUC 与误差指标。

In [ ]:
baselines, failures, results, comparisons = {}, [], [], {}
for name,task in cases:
    case = f'{name}-{task}'
    print('基线',case,flush=True)
    try:
        estimator, space, fit_params = make_case(name,task)
        target = 'FPD' if task == 'binary' else '放款金额'
        estimator.fit(train[features],train[target],**fit_params)
        baselines[case] = estimator
    except Exception as error:
        failures.append({'模型':case,'阶段':'基线','异常':repr(error),'堆栈':traceback.format_exc()})
        print(traceback.format_exc(),flush=True)
for task in ['binary','regression']:
    models = {name:model for name,model in baselines.items() if name.endswith('-'+task)}
    if models:
        target = 'FPD' if task == 'binary' else '放款金额'
        comparison = compare_models(models,train[features],train[target],valid[features],valid[target],n_jobs=1)
        comparison = comparison.rename(columns={'测试集':'选参验证集'},level=1)
        comparison.index = pd.MultiIndex.from_tuples([(name,target) for name,_ in comparison.index],names=['模型名称','目标变量'])
        percentages = [column for column in comparison.columns if column[0] in {'KS','AUC','坏样本率'}]
        dataframe2excel(comparison,str(output/('二分类-基线模型对比.xlsx' if task=='binary' else '回归-基线模型对比.xlsx')),index=True,percent_cols=percentages)
        comparisons[task] = comparison
        display(comparison)

## 5. 独立的预测与报告核对方法

AUC 使用正负样本逐对排序比例计算，KS 使用同分组累计分布计算；误差、R² 与概率均方误差直接用原始行计算。
保存恢复比较整条 Pipeline 的结果，不只保存最后一个学习器。报告工作簿重新打开，检查章节、图片、错误单元格及样本总数。

In [ ]:
def independent_metrics(y,p,task):
    y,p = np.asarray(y),np.asarray(p)
    if task == 'regression':
        residual = p-y
        return {'均方误差':float(np.mean(residual**2)), '均方根误差':float(np.sqrt(np.mean(residual**2))), '平均绝对误差':float(np.mean(np.abs(residual))), '决定系数':float(1-np.sum(residual**2)/np.sum((y-y.mean())**2))}
    pair = p[y==1,None] - p[y==0][None,:]
    auc = np.mean((pair>0)+.5*(pair==0))
    _, groups = np.unique(p,return_inverse=True)
    bad = np.bincount(groups,weights=y)
    good = np.bincount(groups,weights=1-y)
    ks = np.max(np.abs(np.cumsum(bad)/bad.sum()-np.cumsum(good)/good.sum()))
    clipped = np.clip(p,np.finfo(float).eps,1-np.finfo(float).eps)
    return {'AUC':float(auc),'KS':float(ks),'概率均方误差':float(np.mean((p-y)**2)), '交叉熵':float(-np.mean(y*np.log(clipped)+(1-y)*np.log1p(-clipped)))}

def prediction(model,X,task):
    if task == 'regression':
        values = np.asarray(model.predict(X))
        assert values.shape == (len(X),) and np.isfinite(values).all()
        return values
    values = np.asarray(model.predict_proba(X))
    assert values.shape == (len(X),2) and np.isfinite(values).all()
    assert ((values>=0)&(values<=1)).all()
    np.testing.assert_allclose(values.sum(axis=1),1.,rtol=1e-6,atol=1e-7)
    return values[:,1]

def workbook_checks(path):
    workbook = load_workbook(path)
    required = {'目录','1-基本信息','2-模型性能','3-入模变量分析','4-稳定性分析','5-模型参数','6-模型部署需求','7-训练与调参过程'}
    assert required <= set(workbook.sheetnames)
    assert '报告执行状态' not in workbook.sheetnames
    assert all(cell.data_type != 'e' for sheet in workbook for row in sheet for cell in row)
    image_count = sum(len(sheet._images) for sheet in workbook)
    assert image_count >= 3
    sheets = [{'工作表':sheet.title,'行数':sheet.max_row,'列数':sheet.max_column,'图片数':len(sheet._images),'合并区域数':len(sheet.merged_cells.ranges)} for sheet in workbook]
    workbook.close()
    return sheets

for task,comparison in comparisons.items():
    target = 'FPD' if task == 'binary' else '放款金额'
    for case,model in baselines.items():
        if not case.endswith('-'+task):continue
        row = comparison.loc[case].iloc[0]
        for label,frame in [('训练集',train),('选参验证集',valid)]:
            expected = independent_metrics(frame[target],prediction(model,frame[features],task),task)
            required = ['AUC','KS'] if task == 'binary' else ['均方根误差','平均绝对误差','决定系数']
            for metric in required:np.testing.assert_allclose(row[(metric,label)],expected[metric],rtol=1e-6,atol=1e-8)
            assert row[('样本数',label)] == len(frame)

## 6. 每种模型执行真实两点搜索、最终重训与完整报告

最终模型仅重训训练集，不将选参验证或独立测试并入。完整报告包含三组数据，额外保留预测明细、搜索过程、训练过程和模型制品。
若某个模型失败，保存其真实错误并继续检查其余模型；最后的完成断言不会把失败当作成功。

In [ ]:
for name,task in cases:
    case = f'{name}-{task}'
    if case not in baselines:
        continue
    print('全链路',case,flush=True)
    folder = output/case
    folder.mkdir(exist_ok=True)
    started = time.monotonic()
    try:
        estimator, space, fit_params = make_case(name,task)
        target = 'FPD' if task == 'binary' else '放款金额'
        metric = 'accuracy' if name == 'RulesClassifier' else 'auc' if task == 'binary' else 'rmse'
        tuner = hc.ModelTuner(estimator,search_space=space,metric=metric,sampler=optuna.samplers.GridSampler(space,seed=42),fit_params=fit_params,retention='predictions',cv=None,n_jobs=1,random_state=42)
        best_params = tuner.fit(train[features],train[target],eval_set={'选参验证集':(valid[features],valid[target])},n_trials=2,show_progress_bar=False)
        history = tuner.get_optimization_history()
        assert len(tuner.study_.trials) == 2 and all(trial.state.name == 'COMPLETE' for trial in tuner.study_.trials)
        assert len({json.dumps(trial.params,sort_keys=True) for trial in tuner.study_.trials}) == 2
        best = tuner.get_best_model()
        terminal = best.steps[-1][1]
        assert terminal.training_summary_['样本数'] == len(train)
        selection_summary, selection_details = best.named_steps['特征筛选'].report()
        selection = {'筛选汇总': json.loads(selection_summary.to_json(orient='records', force_ascii=False)), '筛选明细': json.loads(selection_details.to_json(orient='records', force_ascii=False))}
        metrics = ['auc','ks','brier','logloss'] if task=='binary' else ['mse','rmse','mae','r2']
        if name=='RulesClassifier':metrics.append('accuracy')
        report = auto_model_report(best,datasets=datasets,target=target,metrics=metrics,explain_config={"enabled":True,"scope":"importance","dataset":"选参验证集","max_samples":32,"max_background":16,"max_evals":2*len(features)+1},model_name=name.replace('-finetune','微调')+('（二分类）' if task=='binary' else '（回归）'),verbose=False,n_jobs=1)
        actual = report.get_metrics().set_index('统计项')
        predictions = []
        independent = {}
        for label,frame in datasets.items():
            p = prediction(best,frame[features],task)
            expected = independent_metrics(frame[target],p,task)
            if name=='RulesClassifier':expected['Accuracy'] = float(np.mean(best.predict(frame[features]) == frame[target].to_numpy()))
            independent[label] = expected
            for key,value in expected.items():
                np.testing.assert_allclose(actual.loc[key,label],value,rtol=1e-6,atol=1e-8)
            assert actual.loc['样本数',label] == len(frame)
            rows = pd.DataFrame({'数据集':label,'原始数据索引':frame.index,'实际值':frame[target].to_numpy(),'预测概率' if task=='binary' else '预测值':p})
            if task=='binary':rows['预测评分' if name in ['ScoreCard','RoundScoreCard','ProbabilityScoreCard'] else '预测类别'] = best.predict(frame[features])
            else:rows['残差'] = p-frame[target].to_numpy()
            predictions.append(rows)
            reordered = frame[features[::-1]].assign(无关字段=1)
            np.testing.assert_allclose(prediction(best,reordered,task),p,rtol=1e-6,atol=1e-7)
            np.testing.assert_allclose(prediction(best,frame.assign(**{target:frame[target].iloc[::-1].to_numpy()}),task),p,rtol=1e-6,atol=1e-7)
        decisions = np.asarray(best.predict(test[features]))
        assert decisions.shape == (len(test),) and np.isfinite(decisions).all()
        if task == 'binary':
            np.testing.assert_array_equal(best.classes_,[0,1])
            if name in ['ScoreCard','RoundScoreCard','ProbabilityScoreCard']:
                p = prediction(best,test[features],task)
                expected_score = terminal.A_ - terminal.B_*np.log(np.clip(p,1e-12,1-1e-12)/(1-np.clip(p,1e-12,1-1e-12)))
                expected_score = np.clip(expected_score,terminal.lower if terminal.lower is not None else -np.inf,terminal.upper if terminal.upper is not None else np.inf)
                tolerance = (len(features)+1)*.5*10**(-terminal.decimal)
                np.testing.assert_allclose(decisions,expected_score,rtol=0,atol=tolerance+1e-6)
            else:
                assert np.isin(decisions,[0,1]).all()
                if name != 'RulesClassifier':np.testing.assert_array_equal(decisions,prediction(best,test[features],task)>=.5)
        sample_path = folder/'预测核验.csv'
        pd.concat(predictions,ignore_index=True).to_csv(sample_path,index=False,encoding='utf-8-sig')
        embedding = None
        if hasattr(terminal,'get_embeddings'):
            try:
                embedding = terminal.get_embeddings(best[:-1].transform(test[features]))
                assert embedding.shape[0] == len(test) and np.isfinite(embedding).all()
                embedding = list(embedding.shape)
            except NotImplementedError:
                embedding = '当前官方权重不提供 embedding'
        path = folder/'完整模型报告.xlsx'
        status = report.to_excel(path,project_desc=report_description(task),data_source=report_source,amount='放款金额',date='放款时间',group='商品类别',with_plots=True,loc_cols='客户编号',model_name=name.replace('-finetune','微调')+('（二分类）' if task=='binary' else '（回归）'),mode='strict',return_result=True)
        assert status.complete
        sheets = workbook_checks(path)
        if name in ['ScoreCard','RoundScoreCard','ProbabilityScoreCard']:
            assert report._is_scorecard_model(), 'Pipeline 中评分卡未被报告识别'
        process = report.get_training_process()
        assert '调参记录' in process and len(process['调参记录']) == 2
        assert '最终重训' in process and '筛选汇总' in process and '预处理步骤' in process
        assert not process['筛选汇总'].empty and '步骤' in process['筛选汇总']
        saved = atomic_save_pickle(best,folder/'最终模型.pkl',engine='cloudpickle')
        restored = load_verified_pickle(saved)
        np.testing.assert_allclose(prediction(restored,test[features],task),prediction(best,test[features],task),rtol=1e-6,atol=1e-7)
        np.testing.assert_allclose(restored.predict(test[features]),decisions,rtol=1e-6,atol=1e-7)
        history.to_csv(folder/'搜索过程.csv',index=False,encoding='utf-8-sig')
        info = {'模型':case,'任务':task,'最佳参数':best_params,'训练样本数':len(train),'验证样本数':len(valid),'独立测试样本数':len(test),'特征筛选':selection,'独立指标':independent,'embedding':embedding,'报告完整':status.complete,'工作表':sheets,'报告':str(path),'模型文件':saved,'耗时秒':round(time.monotonic()-started,2)}
        (folder/'验收.json').write_text(json.dumps(info,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
        results.append(info)
        print('通过',case,info['耗时秒'],flush=True)
        del restored,best,tuner,report,terminal
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    except Exception as error:
        failures.append({'模型':case,'阶段':'全链路','异常':repr(error),'堆栈':traceback.format_exc()})
        print(traceback.format_exc(),flush=True)
    (output/'本次执行.json').write_text(json.dumps({'环境':environment,'计划案例':cases,'通过':[r['模型'] for r in results],'失败':failures},ensure_ascii=False,indent=2,default=str),encoding='utf-8')

## 7. 组合模型、概率校准和评分转换分支

这些组件使用已经训练的模型或概率，不重复伪装成独立预测算法。分别核对其公开 fit/transform/predict、单调方向、输出范围与制品恢复。
CalibratedModel 复用已完成搜索的校准模型，另外导出完整报告。

In [ ]:
components = []
if not chosen:
    try:
        lr = load_verified_pickle(output/'LogisticRegression-binary/最终模型.pkl')
        train_p,valid_p,test_p = [prediction(lr,frame[features],'binary') for frame in (train,valid,test)]
        component_folder = output/'组合组件'
        component_folder.mkdir(exist_ok=True)
        for name in ['PlattCalibrator','IsotonicCalibrator','BetaCalibrator','HistogramCalibrator']:
            obj = getattr(hc,name)().fit(y_prob=valid_p,y_true=valid.FPD)
            values = obj.calibrate(test_p)
            assert values.shape == test_p.shape and np.isfinite(values).all() and ((values>=0)&(values<=1)).all()
            path = atomic_save_pickle(obj,component_folder/f'{name}.pkl',engine='cloudpickle')
            np.testing.assert_allclose(load_verified_pickle(path).calibrate(test_p),values)
            components.append({'组件':name,'样本数':len(values)})
        for method,name in [('standard','StandardScoreTransformer'),('linear','LinearScoreTransformer'),('quantile','QuantileScoreTransformer'),('boxcox','BoxCoxScoreTransformer')]:
            for direction in ['descending','ascending']:
                options = dict(lower=300,upper=900,decimal=4,direction=direction)
                for label,obj in [(name+'-'+direction,getattr(hc,name)(**options)),(f'ScoreTransformer-{method}-{direction}',hc.ScoreTransformer(method=method,**options))]:
                    obj.fit(train_p)
                    values = obj.predict(test_p)
                    assert values.shape == test_p.shape and np.isfinite(values).all()
                    change = np.diff(values[np.argsort(test_p,kind='stable')])
                    assert ((-change if direction=='descending' else change)>=-1e-7).all()
                    assert ((values>=300)&(values<=900)).all()
                    if method == 'linear':
                        expected = 900-600*test_p if direction=='descending' else 300+600*test_p
                        np.testing.assert_allclose(values,expected,rtol=0,atol=.00005+1e-8)
                    path = obj.save(component_folder/f'{label}.pkl')
                    np.testing.assert_allclose(type(obj).load(path).predict(test_p),values)
                    components.append({'组件':label,'样本数':len(values)})
        terminal = lr.steps[-1][1]
        tr,va,te = [lr[:-1].transform(frame[features]) for frame in (train,valid,test)]
        before = terminal.predict_proba(te).copy()
        for method in ['linear','quantile','binning']:
            obj = hc.ScoreDriftCalibrator(method=method).fit(terminal,va,valid.FPD,X_reference=tr,y_reference=train.FPD)
            values = obj.predict_score(te)
            assert values.shape == (len(test),) and np.isfinite(values).all()
            np.testing.assert_allclose(terminal.predict_proba(te),before)
            path = obj.save(component_folder/f'ScoreDriftCalibrator-{method}.pkl')
            np.testing.assert_allclose(hc.ScoreDriftCalibrator.load(path).predict_score(te),values)
            components.append({'组件':f'ScoreDriftCalibrator-{method}','样本数':len(values)})
        truth = pd.DataFrame({'甲':[0,1,0,1],'乙':[0,0,1,1]})
        for logic,expected in [('and',[0,0,0,1]),('or',[0,1,1,1])]:
            obj = hc.RuleSet(rules=[Rule('甲>0'),Rule('乙>0')],logic=logic,n_jobs=1)
            values,details = obj.evaluate(truth)
            np.testing.assert_array_equal(values,expected)
            assert len(details) == 2
            components.append({'组件':f'RuleSet-{logic}','样本数':4})
        calibrated = load_verified_pickle(output/'ProbabilityCalibrator-platt-binary/最终模型.pkl')
        calibration = calibrated.steps[-1][1]
        wrapper = hc.CalibratedModel(calibration.model_,calibration)
        combined = Pipeline(calibrated.steps[:-1]+[('模型',wrapper)])
        np.testing.assert_allclose(prediction(combined,test[features],'binary'),prediction(calibrated,test[features],'binary'))
        path = component_folder/'CalibratedModel-完整模型报告.xlsx'
        status = auto_model_report(combined,datasets=datasets,target='FPD',explain_config={'enabled':True,'scope':'importance','dataset':'选参验证集','max_samples':32,'max_background':16,'max_evals':2*len(combined.feature_names_in_)+1},excel_path=path,project_desc=report_description('binary'),data_source=report_source,amount='放款金额',date='放款时间',group='商品类别',with_plots=True,mode='strict',return_result=True,verbose=False,n_jobs=1)
        assert status.complete
        workbook_checks(path)
        components.append({'组件':'CalibratedModel','样本数':len(test),'报告':str(path)})
    except Exception as error:
        failures.append({'模型':'组合组件','阶段':'组合分支','异常':repr(error),'堆栈':traceback.format_exc()})
        print(traceback.format_exc(),flush=True)
display(pd.DataFrame(components))
(output/'组合组件验收.json').write_text(json.dumps(components,ensure_ascii=False,indent=2),encoding='utf-8')
(output/'本次执行.json').write_text(json.dumps({'环境':environment,'计划案例':cases,'通过':[r['模型'] for r in results],'组合组件':components,'失败':failures},ensure_ascii=False,indent=2,default=str),encoding='utf-8')

## 8. 原生类别特征补充验收

向约定的三个数值字段加入商品类别，验证 CatBoost 和四种 net 的类别输入、预测、embedding 与完整报告。
相关性只针对数值字段，类别变量保留频率、众数和分箱；这组接口检查不参与前面的模型选择。

In [ ]:
categorical_checks = []
if not chosen:
    for name in ['CatBoost','ResNet','FTTransformer','TabM','TabPFN','TabNet']:
        try:
            pipeline,_,_ = make_case(name,'binary')
            estimator = pipeline.steps[-1][1]
            if name in ['ResNet','FTTransformer','TabM','TabNet']:
                estimator.set_params(max_epochs=3,early_stopping_rounds=None)
            columns = features+['商品类别']
            options = {'cat_features':['商品类别']} if name=='CatBoost' else {}
            estimator.fit(train[columns],train.FPD,**options)
            p = prediction(estimator,test[columns],'binary')
            expected = independent_metrics(test.FPD,p,'binary')
            folder = output/(name+'-类别输入')
            folder.mkdir(exist_ok=True)
            report = auto_model_report(estimator,datasets=datasets,target='FPD',explain_config={"enabled":True,"scope":"importance","dataset":"选参验证集","max_samples":32,"max_background":16,"max_evals":2*len(columns)+1},metrics=['auc','ks','brier','logloss'],verbose=False,n_jobs=1)
            table = report.get_metrics().set_index('统计项')
            for metric,value in expected.items():np.testing.assert_allclose(table.loc[metric,'独立测试集'],value,rtol=1e-6,atol=1e-8)
            assert set(report.get_features_describe().index)==set(columns)
            assert '商品类别' not in report.get_features_corr().columns
            path = folder/'完整模型报告.xlsx'
            result = report.to_excel(path,project_desc=report_description('binary'),data_source=report_source,amount='放款金额',date='放款时间',group='商品类别',with_plots=True,loc_cols='客户编号',model_name=name+'（含类别输入）',mode='strict',return_result=True)
            assert result.complete
            workbook_checks(path)
            saved = estimator.save(folder/'最终模型.pkl')
            np.testing.assert_allclose(type(estimator).load(saved).predict_proba(test[columns])[:,1],p,rtol=1e-6,atol=1e-7)
            if hasattr(estimator,'get_embeddings'):
                try:
                    embedding = estimator.get_embeddings(test[columns])
                    assert embedding.shape[0]==len(test) and np.isfinite(embedding).all()
                except NotImplementedError:pass
            categorical_checks.append({'模型':name,'特征数':len(columns),'报告':str(path),'样本数':len(test)})
        except Exception as error:
            failures.append({'模型':name+'-类别输入','阶段':'类别输入','异常':repr(error),'堆栈':traceback.format_exc()})
            print(traceback.format_exc(),flush=True)
    assert len(categorical_checks)==6 or failures
(output/'类别输入验收.json').write_text(json.dumps(categorical_checks,ensure_ascii=False,indent=2),encoding='utf-8')
(output/'本次执行.json').write_text(json.dumps({'环境':environment,'计划案例':cases,'通过':[r['模型'] for r in results],'组合组件':components,'类别输入':categorical_checks,'失败':failures},ensure_ascii=False,indent=2,default=str),encoding='utf-8')

## 9. 汇总实际结果

全模型完成以实际执行清单和逐模型报告为准；任何真实失败都会使最后断言失败。性能指标反映独立测试集，不能用训练集指标替代。

In [ ]:
summary = pd.DataFrame([{'模型':r['模型'],'任务':r['任务'],**r['独立指标']['独立测试集'],'耗时秒':r['耗时秒']} for r in results])
summary['任务'] = summary['任务'].map({'binary':'二分类','regression':'回归'})
display(summary)
summary.to_csv(output/'独立测试指标.csv',index=False,encoding='utf-8-sig')
selection = {}
for task,metric in [('binary','AUC'),('regression','均方根误差')]:
    candidates = [row for row in results if row['任务']==task]
    if candidates:
        winner = (max if task=='binary' else min)(candidates,key=lambda row:row['独立指标']['选参验证集'][metric])
        selection[task] = {'模型':winner['模型'],'选择指标':metric,'验证指标':winner['独立指标']['选参验证集'][metric],'独立测试指标':winner['独立指标']['独立测试集'],'模型文件':winner['模型文件'],'报告':winner['报告']}
(output/'验证集模型选择.json').write_text(json.dumps(selection,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(selection).T)
assert not failures, f'{len(failures)} 个案例失败，详见本次执行.json'
assert len(results) == len(cases)
print(f'本次 {len(results)} 个模型/任务案例全部完成数值与结构验收；样式另按生成工作簿渲染核查。')